<a href="https://colab.research.google.com/github/Antibote/pinn-eos-aluminum/blob/main/notebooks/section5_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Раздел 5. Сжатие и скорость работы (дополнительный этап)

**5.1 Разработка и демонстрация оптимизации решения** — измеримый эффект на размере модели и скорости работы.

Порядок: **раздел 0** → 5.1 → 5.2 → 5.3 → 5.4 (5.4 можно запускать сразу после 5.1). Нужен обученный `artifacts/pinn.pt` (раздел 3).

- **5.1** — исходная точка: PINN 256×4 — размер, время на CPU и GPU, точность;
- **5.2** — дистилляция: сети 128×4 и 64×4 учатся у PINN 256×4 (учитель) с теми же физическими штрафами;
- **5.3** — измерение эффекта: веса float16, ONNX Runtime на CPU, смешанная точность на GPU; таблица и график.
- **5.4** — SVD-сжатие внутренних слоёв 256→256 (вход и выход не сжимаются) + дообучение 1000 шагов с физическими штрафами.
- **5.5** — инференс на C: прямой проход обученной PINN без Python и PyTorch (`csrc/eos_pinn.c`).


## 0. Подготовка (общая, запускать каждому)

In [1]:
import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите GPU в настройках среды')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch 2.11.0+cu130
GPU: Tesla T4


In [ ]:
# Репозиторий: код (src/eos) и папка artifacts — сюда сохраняются все результаты.
# На GitHub их отправляет save_to_repo('что сохранили') — см. ячейку ниже.
import json, subprocess, sys, urllib.parse, urllib.request, zipfile
from pathlib import Path

REPO = Path('pinn-eos-aluminum')
REPO_URL = 'https://github.com/Antibote/pinn-eos-aluminum.git'
_TOKEN = None


def git(*args):
    """git в папке репозитория. Токен в сообщениях об ошибке скрыт."""
    r = subprocess.run(['git', '-C', str(REPO), *args], capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        raise RuntimeError(f'git {args[0]}: ' + (msg.replace(_TOKEN, '***') if _TOKEN else msg))
    return r.stdout


if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
git('config', 'user.name', 'Colab')
git('config', 'user.email', 'colab@users.noreply.github.com')
git('pull', '-q', '--rebase', '--autostash')
sys.path.insert(0, str(REPO / 'src'))

ARTIFACTS = REPO / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Артефакты:', ARTIFACTS.resolve())


In [ ]:
def save_to_repo(message):
    """Коммитит папку artifacts и отправляет её на GitHub.

    Нужен секрет Colab GITHUB_TOKEN (значок ключа слева) с правом записи в репозиторий.
    Без него функция только сообщает, где лежат файлы, — ячейки ноутбука не падают.

    Args:
        message: что сохранили, например 'Бейзлайн: метрики и карта ошибок'.
    """
    global _TOKEN
    try:
        from google.colab import userdata
        _TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:  # не Colab, секрета нет или нет доступа к нему
        _TOKEN = None
    if not _TOKEN:
        print(f'GITHUB_TOKEN не задан — результаты сохранены только в {ARTIFACTS.resolve()} (на GitHub не отправлены)')
        return

    big = [str(p) for p in ARTIFACTS.rglob('*') if p.is_file() and p.stat().st_size > 50 * 2**20]
    if big:
        raise RuntimeError(f'Файлы больше 50 МБ GitHub не примет: {big}')
    git('add', 'artifacts')
    changed = git('diff', '--cached', '--name-only', '--', 'artifacts').split()
    if not changed:
        print('Нечего сохранять: artifacts не изменилась')
        return

    # Автор коммита — владелец токена
    req = urllib.request.Request('https://api.github.com/user', headers={'Authorization': f'token {_TOKEN}'})
    me = json.load(urllib.request.urlopen(req))
    author = f"{me.get('name') or me['login']} <{me['id']}+{me['login']}@users.noreply.github.com>"

    git('commit', '-q', '-m', message, '--author', author)
    url = REPO_URL.replace('https://', f'https://x-access-token:{_TOKEN}@')
    git('pull', '-q', '--rebase', url, 'main')
    git('push', '-q', url, 'HEAD:main')
    print('На GitHub:', git('log', '-1', '--format=%h %s').strip())
    print('Файлы:', *changed, sep='\n  ')


In [ ]:
# Данные с Яндекс Диска (в репозиторий не попадают)
DATA = REPO / 'data'
if not list(DATA.glob('Obl*.txt')):
    api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?' + urllib.parse.urlencode(
        {'public_key': 'https://disk.yandex.ru/d/iUSbYrJcsbXSbw', 'path': '/Материалы к хакатону.zip'})
    href = json.load(urllib.request.urlopen(api))['href']
    urllib.request.urlretrieve(href, 'data.zip')
    zipfile.ZipFile('data.zip').extractall(DATA)
print(len(list(DATA.glob('Obl*.txt'))), 'файлов данных в', DATA)


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eos.data import load_all, region_grid

df = load_all(DATA)
print(df.shape)
df.head()

(1130053, 5)


,rho,T,P,E,region
0,0.001000,0.1,0.000515,-46.797895,Obl01
1,0.001030,0.1,0.000540,-46.790300,Obl01
2,0.001060,0.1,0.000569,-46.782552,Obl01
3,0.001091,0.1,0.000597,-46.774662,Obl01
4,0.001124,0.1,0.000626,-46.766630,Obl01


---
## 5. Сжатие и скорость работы
- [x] SVD-сжатие внутренних слоёв 256→256 (r = 64, 32) + дообучение с физическими штрафами
- [x] Инференс на C: прямой проход без Python и PyTorch, задержка одного запроса и встраивание в расчётные коды


### 5.1 Исходная точка

In [ ]:
# 5.1 Исходная точка: PINN 256×4 из раздела 3 — размер, скорость, точность
import importlib, json, subprocess, sys, torch
import eos.normalize as nz, eos.model as em, eos.pinn as pn, eos.metrics as mt, eos.baseline as bl, eos.inference as inf
for m in (nz, em, mt, pn, bl, inf):
    importlib.reload(m)
try:
    import onnx, onnxruntime
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'], check=True)
pd.set_option('display.width', 250)

NORM = ARTIFACTS / 'normalization.json'
split = dict(np.load(ARTIFACTS / 'split.npz'))
test_df = df.iloc[split['test']]
teacher = inf.load_eosnet(ARTIFACTS / 'pinn.pt', NORM, device=DEVICE)
for p in teacher.parameters():
    p.requires_grad_(False)

def flops_per_query(net):
    """Умножений-сложений на один запрос (по слоям Linear) — от этого зависит время на CPU."""
    return sum(2 * m.in_features * m.out_features for m in net.modules() if isinstance(m, torch.nn.Linear))

with open(ARTIFACTS / 'metrics_pinn.json') as f:
    start = json.load(f)
cpu_start = bl.benchmark(inf.EOSModel(ARTIFACTS / 'pinn.pt', NORM, device='cpu'), n=100_000)
print(f"PINN {teacher.config}: файл {(ARTIFACTS / 'pinn.pt').stat().st_size / 2**20:.3f} МБ, "
      f"{flops_per_query(teacher) / 1e3:.0f} тыс. операций на запрос")
print(f"Время на 10^5 запросов: CPU {cpu_start:.3f} с, GPU {start['time_1e5_s']:.4f} с; "
      f"ошибка P {start['err_P_median_%']:.3f}%, e {start['err_e_median_%']:.3f}% (медианы)")
print('Почти всё время на CPU — умножение матриц 256×256: его и уменьшаем (5.2), плюс ускоряем сам расчёт (5.3)')

### 5.2 Дистилляция в меньшие сети

In [ ]:
# 5.2 Дистилляция: маленькие сети учатся у PINN 256×4 (учитель) + те же физические штрафы.
# Учитель даёт ответы в любых точках области, поэтому ученик учится и между узлами таблицы
STUDENTS = {'pinn_128x4': (128, 4), 'pinn_64x4': (64, 4)}
STEPS_KD = 30_000                       # для пробного запуска — 3000
LAM = {'mono_rho': 1, 'mono_T': 1, 'sound': 1}
LR = 2e-3                               # если loss = nan — уменьшите до 5e-4

def to_t(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE)

trainer = pn.Trainer(*(to_t(df[c].values) for c in ['rho', 'T', 'P', 'e']),
                     idx_train=torch.tensor(split['train'], device=DEVICE),
                     idx_val=torch.tensor(split['val'], device=DEVICE),
                     w_train=to_t(em.region_weights(df.iloc[split['train']])))
students = {}
for name, (hidden, layers) in STUDENTS.items():
    print(f'=== {name}: {hidden}×{layers}')
    torch.manual_seed(0)
    net = em.EOSNet(teacher.norm, hidden=hidden, layers=layers, activation='gelu').to(DEVICE)
    hist = trainer.fit(net, lam=LAM, steps=STEPS_KD, n_coll=1024, lr=LR, teacher=teacher)
    inf.save_net(net, ARTIFACTS / f'{name}.pt')
    students[name] = net
    print(f"    обучение {hist['time_s'] / 60:.1f} мин")
save_to_repo('5.2: дистилляция PINN в сети 128×4 и 64×4')

### 5.3 Измеримый эффект оптимизации

In [ ]:
# 5.3 Измеримый эффект оптимизаций: размер, скорость (CPU / GPU), точность и физика на test
(ARTIFACTS / 'optimized').mkdir(exist_ok=True)
nets = {'PINN 256×4': teacher, 'PINN 128×4 (дистилл.)': students['pinn_128x4'],
        'PINN 64×4 (дистилл.)': students['pinn_64x4']}
rows = []
for name, net in nets.items():
    tag = name.split()[1].replace('×', 'x')
    fp16_path = ARTIFACTS / 'optimized' / f'pinn_{tag}_fp16.pt'
    onnx_path = ARTIFACTS / 'optimized' / f'pinn_{tag}.onnx'
    row = {'модель': name, 'параметров': sum(p.numel() for p in net.parameters()),
           'файл float32, МБ': inf.save_net(net, ARTIFACTS / 'optimized' / f'pinn_{tag}.pt'),
           'файл float16, МБ': inf.save_net(net, fp16_path, fp16=True)}
    # точность и нарушения — у float16-версии (её и отдаём), считаем во float64
    model16 = inf.EOSModel(fp16_path, NORM, device=DEVICE, dtype=torch.float64)
    met, _ = mt.evaluate(model16.predict, test_df, name)
    row.update({k: met[k] for k in ['err_P_median_%', 'err_P_p99_%', 'err_e_median_%', 'err_e_p99_%',
                                    'viol_dP_drho_%', 'viol_de_dT_%', 'viol_cs_%']})
    # скорость на 10^5 запросов
    row['CPU torch, с'] = bl.benchmark(inf.EOSModel(fp16_path, NORM, device='cpu'), n=100_000)
    inf.export_onnx(net, onnx_path)
    row['CPU ONNX Runtime, с'] = bl.benchmark(inf.OnnxEOS(onnx_path), n=100_000)
    if torch.cuda.is_available():
        row['GPU float32, с'] = bl.benchmark(inf.EOSModel(fp16_path, NORM, device='cuda'), n=100_000)
        fast = inf.GpuFp16EOS(net)
        row['GPU float16, с'] = bl.benchmark(fast, n=100_000)
        m16, _ = mt.evaluate(fast.predict, test_df.iloc[::10], name)   # точность при счёте во float16
        row['GPU float16: ошибка P, медиана %'] = m16['err_P_median_%']
    rows.append(row)

with open(ARTIFACTS / 'metrics_baseline.json') as f:
    base = json.load(f)
rows.insert(0, {'модель': 'Бейзлайн (таблица)', 'файл float32, МБ': base['size_MB'],
                'err_P_median_%': base['err_P_median_%'], 'err_P_p99_%': base['err_P_p99_%'],
                'err_e_median_%': base['err_e_median_%'], 'err_e_p99_%': base['err_e_p99_%'],
                'viol_dP_drho_%': base['viol_dP_drho_%'], 'viol_de_dT_%': base['viol_de_dT_%'],
                'CPU torch, с': base['time_1e5_s']})
opt = pd.DataFrame(rows).set_index('модель')
opt.to_csv(ARTIFACTS / 'metrics_optimization.csv')
display(opt.T.round(4))

# Эффект относительно исходной PINN 256×4 float32 на CPU
ref = opt.loc['PINN 256×4']
best_cpu = opt[['CPU torch, с', 'CPU ONNX Runtime, с']].min(axis=1)
print('Эффект оптимизации (относительно PINN 256×4, float32, CPU torch):')
for name in nets:
    print(f"  {name:24s} размер ×{ref['файл float32, МБ'] / opt.loc[name, 'файл float16, МБ']:.1f} меньше, "
          f"CPU ×{ref['CPU torch, с'] / best_cpu[name]:.1f} быстрее, "
          f"ошибка P {opt.loc[name, 'err_P_median_%']:.3f}% (было {ref['err_P_median_%']:.3f}%)")

# График: размер, время на CPU и ошибка
names = list(opt.index)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].bar(names, opt['файл float32, МБ'], color='0.7', label='float32')
ax[0].bar(names, opt['файл float16, МБ'].fillna(opt['файл float32, МБ']), color='C0', label='float16')
ax[0].axhline(10, color='r', ls='--', label='лимит 10 МБ')
ax[0].set(yscale='log', ylabel='МБ', title='Размер модели')
ax[1].bar(names, opt['CPU torch, с'], color='0.7', label='PyTorch')
ax[1].bar(names, best_cpu.fillna(opt['CPU torch, с']), color='C1', label='лучшее (PyTorch / ONNX)')
ax[1].set(yscale='log', ylabel='с на 10^5 запросов', title='Скорость на CPU')
ax[2].bar(names, opt['err_P_median_%'], color='C2', label='P')
ax[2].bar(names, opt['err_e_median_%'], color='C3', alpha=0.6, label='e')
ax[2].set(yscale='log', ylabel='медианная ошибка, %', title='Точность (test)')
for a in ax:
    a.tick_params(axis='x', rotation=30); a.grid(axis='y', alpha=0.3); a.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / 'optimization.png', dpi=130, bbox_inches='tight')
plt.show()
save_to_repo('5.3: оптимизация — размер, скорость, точность')

### 5.4 SVD-сжатие внутренних слоёв + дообучение

In [ ]:
# 5.4 SVD-сжатие внутренних слоёв + дообучение с физическими штрафами («безопасный путь»)
# Внутренние слои 256→256 заменяются произведением матриц ранга r: W ≈ (U_r·S_r)·V_rᵀ (256→r→256).
# Первый слой 2→256 и последний 256→2 не сжимаются — они маленькие и критичны для физики.
# Обрезка ранга сразу портит точность (веса не низкоранговые), поэтому дальше — короткое дообучение
# с теми же физическими штрафами и исходной PINN в роли учителя.
RANKS = [64, 32]            # r = 64 — сжатие внутренних слоёв в 2 раза, r = 32 — в 4 раза
STEPS_FT = 1000             # 500–1000 шагов дообучения
LAM = {'mono_rho': 1, 'mono_T': 1, 'sound': 1}

if 'trainer' not in globals():   # trainer создаётся в 5.2; если 5.2 не запускали — создаём здесь
    def to_t(a):
        return torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE)
    trainer = pn.Trainer(*(to_t(df[c].values) for c in ['rho', 'T', 'P', 'e']),
                         idx_train=torch.tensor(split['train'], device=DEVICE),
                         idx_val=torch.tensor(split['val'], device=DEVICE),
                         w_train=to_t(em.region_weights(df.iloc[split['train']])))
(ARTIFACTS / 'optimized').mkdir(exist_ok=True)

def test_metrics(net):
    met, _ = mt.evaluate(inf.EOSModel.from_net(net, device=DEVICE, dtype=torch.float64).predict, test_df, '')
    return {k: met[k] for k in ['err_P_median_%', 'err_P_p99_%', 'err_e_median_%', 'err_e_p99_%',
                                'viol_dP_drho_%', 'viol_de_dT_%', 'viol_cs_%']}

rows = [{'модель': 'PINN 256×4 (исходная)', 'параметров': sum(p.numel() for p in teacher.parameters()),
         'файл float16, МБ': inf.save_net(teacher, ARTIFACTS / 'optimized' / 'pinn_256x4_fp16.pt', fp16=True),
         **test_metrics(teacher),
         'CPU torch, с': bl.benchmark(inf.EOSModel.from_net(teacher, device='cpu'), n=100_000)}]
svd_nets = {}
for r in RANKS:
    print(f'=== SVD r = {r}: доля Σσ² в первых r сингулярных числах по слоям — {np.round(inf.svd_energy(teacher, r), 3)}')
    net = inf.compress_svd(teacher, r)
    for p in net.parameters():
        p.requires_grad_(True)
    rows.append({'модель': f'SVD r={r}, без дообучения', 'параметров': sum(p.numel() for p in net.parameters()),
                 **test_metrics(net)})
    hist = trainer.fit(net, lam=LAM, steps=STEPS_FT, n_coll=1024, lr=5e-4, ramp=0.0, teacher=teacher, eval_every=250)
    path = ARTIFACTS / 'optimized' / f'pinn_svd{r}_fp16.pt'
    row = {'модель': f'SVD r={r} + дообучение', 'параметров': sum(p.numel() for p in net.parameters()),
           'файл float16, МБ': inf.save_net(net, path, fp16=True), **test_metrics(net),
           'CPU torch, с': bl.benchmark(inf.EOSModel(path, NORM, device='cpu'), n=100_000),
           'дообучение, с': hist['time_s']}
    inf.export_onnx(net, ARTIFACTS / 'optimized' / f'pinn_svd{r}.onnx')
    row['CPU ONNX Runtime, с'] = bl.benchmark(inf.OnnxEOS(ARTIFACTS / 'optimized' / f'pinn_svd{r}.onnx'), n=100_000)
    if torch.cuda.is_available():
        row['GPU float32, с'] = bl.benchmark(inf.EOSModel(path, NORM, device='cuda'), n=100_000)
    rows.append(row)
    svd_nets[r] = net

svd_table = pd.DataFrame(rows).set_index('модель')
svd_table.to_csv(ARTIFACTS / 'metrics_svd.csv')
display(svd_table.T.round(4))

ref = svd_table.iloc[0]
for r in RANKS:
    row = svd_table.loc[f'SVD r={r} + дообучение']
    print(f"SVD r={r}: параметров ×{ref['параметров'] / row['параметров']:.1f} меньше, "
          f"файл {row['файл float16, МБ']:.3f} МБ (×{ref['файл float16, МБ'] / row['файл float16, МБ']:.1f}), "
          f"ошибка P {row['err_P_median_%']:.3f}% (было {ref['err_P_median_%']:.3f}%), "
          f"нарушения ∂P/∂ρ {row['viol_dP_drho_%']:.3f}%, ∂e/∂T {row['viol_de_dT_%']:.3f}%, c_s {row['viol_cs_%']:.3f}%")

# График: ошибка до и после дообучения, размер
names = list(svd_table.index)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(names, svd_table['err_P_median_%'], color='C0', label='P')
ax[0].bar(names, svd_table['err_e_median_%'], color='C3', alpha=0.6, label='e')
ax[0].set(yscale='log', ylabel='медианная ошибка на test, %', title='SVD: точность до и после дообучения')
sizes = svd_table['файл float16, МБ'].dropna()
ax[1].bar(sizes.index, sizes, color='C1')
ax[1].set(ylabel='МБ (веса float16)', title='Размер файла')
for a in ax:
    a.tick_params(axis='x', rotation=30); a.grid(axis='y', alpha=0.3)
ax[0].legend()
plt.tight_layout()
plt.savefig(FIGURES / 'svd_compression.png', dpi=130, bbox_inches='tight')
plt.show()
save_to_repo('5.4: SVD-сжатие внутренних слоёв + дообучение с физическими штрафами')

### 5.5 Инференс на C (прямой проход без Python и PyTorch)

In [ ]:
# 5.5 Инференс на C: прямой проход обученной PINN без Python и PyTorch (код — csrc/eos_pinn.c, src/eos/cinfer.py)
# Обучение и autograd остаются в PyTorch; на C — только нормировка → слои Linear + GELU → обратная нормировка.
# Зачем: уравнение состояния вызывают из расчётных кодов (C/Fortran) миллионы раз по одной точке на ячейку —
# там важна задержка одного вызова и отсутствие зависимостей, а не пакет из 10^5 точек на GPU.
import importlib, json, subprocess, sys, time
import numpy as np, pandas as pd, torch
import eos.normalize as nz, eos.model as em, eos.metrics as mt, eos.baseline as bl, eos.inference as inf, eos.cinfer as ci
for m_ in (nz, em, mt, bl, inf, ci):
    importlib.reload(m_)
try:
    import onnxruntime
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'], check=True)
NORM = ARTIFACTS / 'normalization.json'                       # всё нужное — здесь: ячейку можно запускать сразу после раздела 0
test_df = df.iloc[np.load(ARTIFACTS / 'split.npz')['test']]

C_DIR = ARTIFACTS / 'c'
C_DIR.mkdir(exist_ok=True)
models_c = {'PINN 256×4': ARTIFACTS / 'pinn.pt', 'PINN SVD r=64': ARTIFACTS / 'optimized' / 'pinn_svd64_fp16.pt'}
lib = ci.build()
exe = ci.CSRC / 'build' / 'example'
subprocess.run(['gcc', '-O3', '-march=native', '-ffast-math', '-fopenmp', str(ci.CSRC / 'example.c'),
                str(ci.CSRC / 'eos_pinn.c'), '-o', str(exe), '-lm'], check=True)

def latency(f, n=2000):
    """Средняя задержка одного запроса, мкс."""
    pts = test_df[['rho', 'T']].to_numpy()[:n]
    f(pts[0, :1], pts[0, 1:])
    t = time.perf_counter()
    for r, T_ in pts:
        f(np.array([r]), np.array([T_]))
    return (time.perf_counter() - t) / n * 1e6

rows = []
for name, path in models_c.items():
    net = inf.load_eosnet(path, NORM)
    bin_path = C_DIR / (path.stem + '.bin')
    size = ci.export_weights(net, bin_path)
    cm = ci.CEOS(bin_path)
    tm = inf.EOSModel.from_net(net, device='cpu')
    onnx_path = C_DIR / (path.stem + '.onnx'); inf.export_onnx(net, onnx_path); om = inf.OnnxEOS(onnx_path)
    # точность C относительно PyTorch и на test
    P_t, e_t = inf.EOSModel.from_net(net, device='cpu', dtype=torch.float64).predict(test_df['rho'].values, test_df['T'].values)
    P_c, e_c = cm.predict(test_df['rho'].values, test_df['T'].values)
    met, _ = mt.evaluate(cm.predict, test_df, name)
    # чистый C (без Python): задержка одного вызова и пакет 10^5
    out = subprocess.run([str(exe), str(bin_path)], capture_output=True, text=True, check=True).stdout
    c_one = float(out.split('eos_predict_one:')[1].split()[0]); c_batch = float(out.split('10^5 точек:')[1].split()[0])
    rows.append({'модель': name, 'файл весов C, МБ': size,
                 'макс. отличие P от PyTorch': float(np.max(np.abs(P_c / P_t - 1))),
                 'ошибка P на test, медиана %': met['err_P_median_%'], 'ошибка e на test, медиана %': met['err_e_median_%'],
                 '10^5 запросов: PyTorch CPU, с': bl.benchmark(tm, n=100_000),
                 '10^5 запросов: ONNX Runtime, с': bl.benchmark(om, n=100_000),
                 '10^5 запросов: C (из Python), с': bl.benchmark(cm, n=100_000),
                 '10^5 запросов: чистый C, с': c_batch,
                 '1 запрос: PyTorch, мкс': latency(tm.predict),
                 '1 запрос: ONNX Runtime, мкс': latency(om.predict),
                 '1 запрос: C (из Python), мкс': latency(lambda r, T_: cm.predict_one(r[0], T_[0])),
                 '1 запрос: чистый C, мкс': c_one})
c_table = pd.DataFrame(rows).set_index('модель')
c_table.to_csv(ARTIFACTS / 'metrics_c.csv')
display(c_table.T.round(5))
for name in c_table.index:
    r = c_table.loc[name]
    print(f"{name}: один запрос — чистый C {r['1 запрос: чистый C, мкс']:.1f} мкс против PyTorch {r['1 запрос: PyTorch, мкс']:.0f} мкс "
          f"(×{r['1 запрос: PyTorch, мкс'] / r['1 запрос: чистый C, мкс']:.0f}) и ONNX {r['1 запрос: ONNX Runtime, мкс']:.0f} мкс "
          f"(×{r['1 запрос: ONNX Runtime, мкс'] / r['1 запрос: чистый C, мкс']:.1f}); точность совпадает с PyTorch до "
          f"{r['макс. отличие P от PyTorch']:.0e}")
print('\nВывод программы на чистом C (csrc/example.c):\n' + subprocess.run([str(exe), str(C_DIR / 'pinn.bin')], capture_output=True, text=True).stdout)
save_to_repo('5.5: инференс PINN на C — точность и скорость')